In [1]:
import re
import string
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.sentiment.vader import SentimentIntensityAnalyzer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.naive_bayes import BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

# Unduh dataset pendukung NLTK
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('vader_lexicon')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [2]:
# Memuat dataset hasil scraping
df = pd.read_csv('ulasan_aplikasi.csv')

# Deteksi nama kolom ulasan yang tersedia ('content' atau 'Review')
target_col = 'content' if 'content' in df.columns else 'Review'

# Ambil kolom ulasan, bersihkan, lalu ubah nama kolomnya menjadi 'content'
clean_df = df[[target_col]].dropna().drop_duplicates()
clean_df.columns = ['content']

print(f"Total ulasan bersih yang akan diproses: {clean_df.shape[0]}")
clean_df.head()

Total ulasan bersih yang akan diproses: 13106


,content
0,The negative reviews on this are hilarious 😭 b...
1,"Good as VN: adequate design encoding, historic..."
2,"epic story, epic gacha"
3,This game is Peak! The illustrations are amazi...
4,"I loved this game, I had fun, I genuinely enjo..."


In [3]:
def cleaningText(text):
    text = re.sub(r'@[A-Za-z0-9]+', '', text) # Hapus mention
    text = re.sub(r'#[A-Za-z0-9]+', '', text) # Hapus hashtag
    text = re.sub(r'RT[\s]', '', text)        # Hapus RT
    text = re.sub(r"http\S+", '', text)        # Hapus URL
    text = re.sub(r'[0-9]+', '', text)         # Hapus angka
    text = text.replace('\n', ' ')
    text = text.translate(str.maketrans('', '', string.punctuation)) # Hapus tanda baca
    text = text.strip(' ')
    return text

def casefoldingText(text):
    return text.lower()

def tokenizingText(text):
    return word_tokenize(text)

def filteringText(text_tokens):
    listStopwords = set(stopwords.words('english'))
    return [word for word in text_tokens if word not in listStopwords]

def toSentence(list_words):
    return ' '.join(list_words)

# Menjalankan rantai preprocessing
clean_df['text_clean'] = clean_df['content'].apply(cleaningText)
clean_df['text_casefolding'] = clean_df['text_clean'].apply(casefoldingText)
clean_df['text_tokenizing'] = clean_df['text_casefolding'].apply(tokenizingText)
clean_df['text_stopword'] = clean_df['text_tokenizing'].apply(filteringText)
clean_df['text_akhir'] = clean_df['text_stopword'].apply(toSentence)

clean_df[['content', 'text_akhir']].head()

,content,text_akhir
0,The negative reviews on this are hilarious 😭 b...,negative reviews hilarious 😭 boo hoo
1,"Good as VN: adequate design encoding, historic...",good vn adequate design encoding historical re...
2,"epic story, epic gacha",epic story epic gacha
3,This game is Peak! The illustrations are amazi...,game peak illustrations amazing stages make st...
4,"I loved this game, I had fun, I genuinely enjo...",loved game fun genuinely enjoyed story god lov...


In [4]:
sia = SentimentIntensityAnalyzer()

def sentiment_analysis_vader(text):
    score = sia.polarity_scores(text)['compound']
    polarity = 'positive' if score >= 0 else 'negative'
    return score, polarity

results = clean_df['text_akhir'].apply(sentiment_analysis_vader)
results = list(zip(*results))

clean_df['polarity_score'] = results[0]
clean_df['polarity'] = results[1]

print("Distribusi Sentimen:")
print(clean_df['polarity'].value_counts())

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=300, hidden_dim=256, num_layers=2, bidirectional=True):
        super(LSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional
        )
        num_directions = 2 if bidirectional else 1
        self.fc = nn.Linear(hidden_dim * num_directions, 2) # Output: 2 kelas (positive/negative)

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, (hn, cn) = self.lstm(embedded)
        # Mengambil hidden state terakhir
        if self.lstm.bidirectional:
            last_hidden = torch.cat((hn[-2], hn[-1]), dim=1)
        else:
            last_hidden = hn[-1]
        out = self.fc(last_hidden)
        return out

Distribusi Sentimen:
polarity
positive    12039
negative     1067
Name: count, dtype: int64


In [5]:
X = clean_df['text_akhir']
y = clean_df['polarity']

# Ekstraksi Fitur TF-IDF
tfidf = TfidfVectorizer(max_features=1000, min_df=5, max_df=0.8)
X_tfidf = tfidf.fit_transform(X)

# Pembagian data latih (80%) dan data uji (20%)
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

print(f"Data Latih: {X_train.shape[0]} samples")
print(f"Data Uji  : {X_test.shape[0]} samples")

Data Latih: 10484 samples
Data Uji  : 2622 samples


In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.model_selection import train_test_split

# 1. Konversi label string ('positive'/'negative') ke integer (1/0)
label_map = {'positive': 1, 'negative': 0}
clean_df['label'] = clean_df['polarity'].map(label_map)

# 2. Gunakan TEKS ASLI ('content') untuk Transformer (BUKAN 'text_akhir' yang dikikis stopword)
X_text = clean_df['content'].tolist()
y_labels = clean_df['label'].tolist()

# Pembagian data Train (80%) dan Test (20%) secara Stratified
train_texts, test_texts, train_labels, test_labels = train_test_split(
    X_text, y_labels, test_size=0.2, random_state=42, stratify=y_labels
)

# 3. Load Tokenizer RoBERTa
model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 4. Tokenisasi Data
train_encodings = tokenizer(train_texts, padding="max_length", truncation=True, max_length=128)
test_encodings = tokenizer(test_texts, padding="max_length", truncation=True, max_length=128)

# 5. Buat Dataset PyTorch
class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = SentimentDataset(train_encodings, train_labels)
test_dataset = SentimentDataset(test_encodings, test_labels)

print("Data & Tokenizer Transformer siap untuk proses Fine-Tuning!")

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Data & Tokenizer Transformer siap untuk proses Fine-Tuning!


In [8]:
import numpy as np
from sklearn.metrics import accuracy_score

# 1. Load Pre-trained Model untuk Klasifikasi 2 Kelas (Positive & Negative)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    ignore_mismatched_sizes=True
)

# 2. HYPERPARAMETER TUNING UNTUK DEEP LEARNING
training_args = TrainingArguments(
    output_dir='./results_roberta',
    num_train_epochs=3,                     # Epoch ideal untuk Transformer (3-5)
    per_device_train_batch_size=32,         # Ukuran batch per GPU/CPU
    per_device_eval_batch_size=64,
    fp16=True,                              # Memanfaatkan Tensor Cores GPU T4
    learning_rate=2e-5,                     # Learning rate kecil menjaga bobot pre-trained
    weight_decay=0.01,                      # Regularisasi L2 mencegah overfitting
    warmup_ratio=0.1,                       # Stabilitas konvergensi di awal pelatihan
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=50,
)

# 3. Fungsi Pengukuran Akurasi Evaluasi
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}

# 4. Inisialisasi Trainer Hugging Face
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

# 5. Jalankan Fine-Tuning
print("--- Memulai Fine-Tuning RoBERTa ---")
trainer.train()

# 6. Evaluasi Akurasi Akhir pada Data Uji
eval_results = trainer.evaluate()
roberta_acc = eval_results['eval_accuracy']

print(f"\n==================================================")
print(f"RoBERTa (Fine-Tuned) Test Accuracy : {roberta_acc:.4f}")
print(f"==================================================")

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `3`.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |                                                                                       
----------------------------+------------+---------------------------------------------------------------------------------------
roberta.pooler.dense.bias   | UNEXPECTED |                                                                                       
roberta.pooler.dense.weight | UNEXPECTED |                                                                                       
classifier.out_proj.bias    | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([2])          
classifier.out_proj.weight  | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([3, 768]) vs model:torch.Size([2, 768])

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect 

--- Memulai Fine-Tuning RoBERTa ---


Epoch,Training Loss,Validation Loss,Accuracy
1,0.144564,0.122119,0.954233
2,0.105026,0.129470,0.954615
3,0.060577,0.151208,0.955378


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy
0.060577,0.151208,3,0.955378



RoBERTa (Fine-Tuned) Test Accuracy : 0.9554


In [9]:
# Evaluasi Model Machine Learning Klasik berbasis TF-IDF
ml_models = {
    "Naive Bayes": BernoulliNB(),
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42)
}

ml_results = {}

print("--- Hasil Evaluasi Machine Learning (TF-IDF) ---")
for name, model in ml_models.items():
    # Melatih model dengan data TF-IDF
    model.fit(X_train.toarray(), y_train)

    # Prediksi
    y_pred_test = model.predict(X_test.toarray())
    acc_test = accuracy_score(y_test, y_pred_test)
    ml_results[name] = acc_test

    print(f"{name:20s} | Test Accuracy: {acc_test:.4f}")

--- Hasil Evaluasi Machine Learning (TF-IDF) ---
Naive Bayes          | Test Accuracy: 0.8978
Logistic Regression  | Test Accuracy: 0.9378
Random Forest        | Test Accuracy: 0.9416
Decision Tree        | Test Accuracy: 0.9100


In [12]:
# Rekapitulasi Performa Seluruh Model
summary_data = {
    "Model": ["RoBERTa (Pre-trained)", "Naive Bayes", "Logistic Regression", "Random Forest", "Decision Tree"],
    "Accuracy": [
        roberta_acc,
        ml_results["Naive Bayes"],
        ml_results["Logistic Regression"],
        ml_results["Random Forest"],
        ml_results["Decision Tree"]
    ]
}

df_summary = pd.DataFrame(summary_data)
df_summary = df_summary.sort_values(by="Accuracy", ascending=False).reset_index(drop=True)

print("=== PERBANDINGAN PERFORMA SEMUA MODEL ===")
df_summary

=== PERBANDINGAN PERFORMA SEMUA MODEL ===


,Model,Accuracy
0,RoBERTa (Pre-trained),0.955378
1,Random Forest,0.941648
2,Logistic Regression,0.937834
3,Decision Tree,0.909992
4,Naive Bayes,0.897788


In [15]:
# ==========================================================
# INFERENCE DENGAN MODEL RANDOM FOREST
# ==========================================================

# Ensure preprocessing functions and TF-IDF vectorizer are available from previous cells.
# Functions: cleaningText, casefoldingText, tokenizingText, filteringText, toSentence
# Objects: tfidf, ml_models

def predict_new_review_rf(text_review):
    # 1. Preprocessing the new review using the same steps as training data
    cleaned_text = cleaningText(text_review)
    cased_text = casefoldingText(cleaned_text)
    tokenized_text = tokenizingText(cased_text)
    filtered_text = filteringText(tokenized_text)
    final_text = toSentence(filtered_text)

    # 2. TF-IDF Vectorization
    # The tfidf vectorizer expects an iterable (e.g., list) of strings
    text_vectorized = tfidf.transform([final_text])

    # 3. Prediction using the trained Random Forest model
    rf_model = ml_models["Random Forest"]
    predicted_label = rf_model.predict(text_vectorized)[0] # Get the predicted label string ('positive' or 'negative')

    # 4. Confidence Score (using predict_proba)
    probabilities = rf_model.predict_proba(text_vectorized)[0]

    # Determine the index for 'positive' and 'negative' labels in the probabilities array
    # Based on y_train, 'positive' is the second class, 'negative' is the first.
    neg_idx = np.where(rf_model.classes_ == 'negative')[0][0]
    pos_idx = np.where(rf_model.classes_ == 'positive')[0][0]

    if predicted_label == 'positive':
        confidence = probabilities[pos_idx]
        sentiment_str = "POSITIF"
    else: # predicted_label == 'negative'
        confidence = probabilities[neg_idx]
        sentiment_str = "NEGATIF"

    return sentiment_str, confidence

# --- UJI COBA ULASAN BARU ---
test_reviews = [
    "The storyline is super amazing and the English voice acting is top-tier!",
    "Too many bugs after the update, the game keeps crashing on the loading screen.",
    "The game is okay, average gacha mechanics."
]

print("=== HASIL INFERENCE SENTIMEN (RANDOM FOREST) ===")
for review in test_reviews:
    sentimen, conf = predict_new_review_rf(review)
    print(f"Ulasan   : \"{review}\"")
    print(f"Sentimen : {sentimen} (Confidence: {conf*100:.2f}%)")
    print("-" * 50)

=== HASIL INFERENCE SENTIMEN (RANDOM FOREST) ===
Ulasan   : "The storyline is super amazing and the English voice acting is top-tier!"
Sentimen : POSITIF (Confidence: 99.00%)
--------------------------------------------------
Ulasan   : "Too many bugs after the update, the game keeps crashing on the loading screen."
Sentimen : POSITIF (Confidence: 63.00%)
--------------------------------------------------
Ulasan   : "The game is okay, average gacha mechanics."
Sentimen : POSITIF (Confidence: 88.00%)
--------------------------------------------------
